In [ ]:
# 프로토타입 파이프라인
# 기본 전처리 구성
# lgbm + ridge

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_log_error
import lightgbm as lgb


'''
코드설명
    - 데이터 전처리: 로그 변환, 결측치 처리, 범주형 인코딩
    - 텍스트 처리: TF-IDF로 상품명과 설명을 벡터화
    - 모델 학습: Ridge(텍스트 강점) + LightGBM(범주형 강점)
    - 앙상블: 두 모델의 예측값을 평균 → RMSLE 개선
    - 평가: RMSLE로 성능 측정
'''

# 1. 데이터 로드
data = pd.read_csv("../data/train.tsv", sep="\t")

# 2. 데이터 전처리
# 로그 변환된 가격
data = data.dropna(subset=["price"])
data["price_log"] = np.log1p(data["price"])

# 결측치 처리
data["brand_name"].fillna("missing", inplace=True)
data["item_description"].fillna("missing", inplace=True)

# 3. 텍스트 특징 추출 (TF-IDF)
tfidf_name = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_name = tfidf_name.fit_transform(data["name"])

tfidf_desc = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_desc = tfidf_desc.fit_transform(data["item_description"])

# 4. 범주형 인코딩
le_brand = LabelEncoder()
data["brand_enc"] = le_brand.fit_transform(data["brand_name"])

le_cat = LabelEncoder()
data["cat_enc"] = le_cat.fit_transform(data["category_name"])

# 5. 특징 결합
import scipy.sparse as sp
X_num = sp.csr_matrix(data[["item_condition_id","shipping","brand_enc","cat_enc"]].values)
X = sp.hstack((X_name, X_desc, X_num)).tocsr()
y = data["price_log"]

# 6. 데이터 분할
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. 모델 학습 (Ridge + LightGBM)
ridge = Ridge(alpha=1.0, solver="lsqr")
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_valid)

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_valid = lgb.Dataset(X_valid, label=y_valid)

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.1,
    "num_leaves": 31,
    "verbose": -1,
    "device": "gpu",          # GPU 사용
    "gpu_platform_id": 0,     # GPU 플랫폼 ID
    "gpu_device_id": 0        # GPU 디바이스 ID

}

lgb_model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_valid],
    num_boost_round=100,
    callbacks=[lgb.early_stopping(10)]   # 콜백으로 지정
)
lgb_pred = lgb_model.predict(X_valid)

# 8. 앙상블 (단순 평균)
final_pred = (ridge_pred + lgb_pred) / 2

# 9. 평가 (RMSLE) - 음수 방지 처리
final_pred = np.maximum(final_pred, 0)
rmsle = np.sqrt(mean_squared_log_error(np.expm1(y_valid), np.expm1(final_pred)))
print("Validation RMSLE:", rmsle)

result_prototype='''
[100]	valid_0's rmse: 0.550791
Validation RMSLE: 0.6163456947048415
'''

Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's rmse: 0.550791
Validation RMSLE: 0.6163456947048415


In [ ]:
# Hyper Parameter Tuning
# 1. GridSearchCV(Scikit-Learn API)
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV

# 모델 정의
lgb_model = LGBMRegressor(
    objective="regression",
    device="gpu",   # GPU 사용
    metric="rmse"
)

# 탐색할 하이퍼파라미터 범위
param_grid = {
    "num_leaves": [31, 63, 127],
    "learning_rate": [0.05, 0.1, 0.2],
    "n_estimators": [100, 300, 500],
    "max_depth": [-1, 10, 20]
}

# GridSearchCV 실행
grid = GridSearchCV(
    estimator=lgb_model,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=2
)

grid.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_metric="rmse", early_stopping_rounds=20)

print("Best parameters:", grid.best_params_)
print("Best score:", -grid.best_score_)

In [ ]:
# 2. Optuna(베이지안 최적화 기반)